# HW3P2: Automatic Speech Recognition with an Encoder-Decoder Transformer

# Schedule:
- Kaggle Submission (DUE 6 November 2026 @ 11:59PM EST | Slack Deadline is 13 November 2026 @ 11:59PM EST)
- Code Submission (DUE 8 November 2026 @ 11:59PM EST | Closes 13 November 2026 @ 11:59PM EST, with the Slack Kaggle Competition)


## Requirement Acknowledgement

Setting the flag below to **True** indicates that you have read, understood, and agree to comply with **all** of the following requirements:

1. **Slack Days Usage**  
   Slack days may be used **only** for the **P2 Final submission**.  
   Specifically, slack days may be applied to submit **final P2 Kaggle scores** to the **Slack Kaggle Competition**, at the expense of your available slack days.

2. **Final Code Submission Deadline**  
   The final **Gradescope code submission** is due **within 48 hours after** the Kaggle competition deadline.

3. **Kaggle Username & Score Verification**  
   You must provide your **Kaggle username** below. We will use it to retrieve your official **PRIVATE leaderboard** score.  
   Minor score or output variance is acceptable; however, any discrepancies caused by **modifications to the required submission code**
   (including the notebook’s final submission cell) will result in an **Academic Integrity Violation (AIV)**.

4. **Model Implementation Restrictions**  
   You are **not permitted** to use pre-trained or pre-loaded models (e.g., from Hugging Face or similar libraries).  
   You **may** implement models described in research papers or articles, but they **must** be implemented **from scratch**
   using fundamental PyTorch components (e.g., `Linear`, `Conv2d`, etc.).

5. **Data Usage Restrictions**  
   The use of **any external data or datasets** is strictly prohibited at **any stage** of this assignment.

6. **Collaboration Policy**  
   You may collaborate with teammates to run experiments or ablations.  
   However, you must submit **your own code** and **your own results**.

7. **Academic Integrity**  
   Failure to comply with any of the above requirements will be treated as an **Academic Integrity Violation (AIV)**.

8. **Late Submissions**  
   Late submissions **must** be submitted through the **Slack Kaggle Competition** (see write-up for details).  
   Submissions made to the **regular Kaggle competition** after the original deadline will **not** be considered,
   regardless of remaining slack days.
   
   Gradescope code submission will close at **the same time** as the Slack Kaggle Competition, rather than remaining open for an additional 48 hours.


In [ ]:
# <<< 手动修改: 读完上面的课程声明并同意后，把 False 改成 True >>>
ACKNOWLEDGED = False

# Colab (T4) Setup
This notebook is the Colab-only version of the starter notebook.

**运行时 / Runtime**：代码执行程序 → 更改运行时类型 → **T4 GPU**

**凭证 / Credentials**：所有 API key 和 token 都集中在下面这一个 cell 里。
把每个 `<<< 手动粘贴 ... >>>` 换成你自己的值，其余地方都不用改。
⚠️ 不要把填好凭证的 notebook 上传到 GitHub 或分享给别人。

In [ ]:
# ==================================================================================
# 🔑 手动填写区 / FILL IN MANUALLY — 把下面每个 "<<< ... >>>" 替换成你自己的值
# ==================================================================================
GITHUB_TOKEN     = "<<< 手动粘贴: GitHub token (ghp_ 开头) >>>"
KAGGLE_USERNAME  = "<<< 手动粘贴: Kaggle 用户名 (个人主页网址里的那个, 不能有空格) >>>"
KAGGLE_API_TOKEN = "<<< 手动粘贴: Kaggle API token (KGAT_ 开头) >>>"
WANDB_API_KEY    = "<<< 手动粘贴: W&B API key (wandb_ 开头) >>>"
WANDB_ENTITY     = "<<< 手动粘贴: W&B 用户名/团队名 (W&B 网址 wandb.ai/<这里> 的那一段) >>>"
WANDB_PROJECT    = "HW3P2"
# ==================================================================================

import os
_creds = dict(GITHUB_TOKEN=GITHUB_TOKEN, KAGGLE_USERNAME=KAGGLE_USERNAME,
              KAGGLE_API_TOKEN=KAGGLE_API_TOKEN, WANDB_API_KEY=WANDB_API_KEY, WANDB_ENTITY=WANDB_ENTITY)
_missing = [k for k, v in _creds.items() if v.startswith("<<<")]
assert not _missing, f"还没填: {_missing}"
assert " " not in KAGGLE_USERNAME, "KAGGLE_USERNAME 不能有空格，要用个人主页网址里的用户名"
assert KAGGLE_API_TOKEN.startswith("KGAT_"), "KAGGLE_API_TOKEN 应该以 KGAT_ 开头"

!nvidia-smi --query-gpu=name,memory.total --format=csv

### Step 1: Get your handout (private GitHub repo)

In [ ]:
%cd /content
GITHUB_USERNAME = "BritneyBao"
REPO_NAME       = "IDL-HW3"
TOKEN = GITHUB_TOKEN   # 来自上面的手动填写区
if not os.path.exists(REPO_NAME):
    !git clone -q https://{TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git
else:
    !cd {REPO_NAME} && git pull -q
!ls {REPO_NAME}

### Step 2: Install Dependencies

In [ ]:
!pip install wandb==0.23.1 torchinfo torchmetrics kaggle==1.8.0 "kagglehub>=1.0.0" --quiet

### Step 3: Configure Kaggle API Access
Join the HW3P2 kaggle competition first (https://www.kaggle.com/t/6aedc76c79fa415fb477f967ffd7c799).

In [ ]:
os.environ['KAGGLE_USERNAME']  = KAGGLE_USERNAME    # 来自上面的手动填写区
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_TOKEN   # 来自上面的手动填写区
assert os.environ['KAGGLE_API_TOKEN'].startswith("KGAT_"), "Use the API token that starts with KGAT_"
import kaggle
api = kaggle.api  # Already authenticated on import

### Step 4: Obtain Data
Downloads and unzips to `/content/hw3p2-data/hw3p2_data` (about 5–10 min).

In [ ]:
if not os.path.exists('/content/hw3p2-data/hw3p2_data'):
    api.dataset_download_files("cmu11785/f26-hw3p2-data", path="/content", unzip=True, quiet=False)
!du -h --max-depth=2 /content/hw3p2-data/hw3p2_data

### Step 5: Persist experiments to Google Drive
Free Colab sessions disconnect. Checkpoints are written to `expts/` and that folder is linked to Drive so they survive a disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_EXPTS = '/content/drive/MyDrive/IDL-HW3P2/expts'
os.makedirs(DRIVE_EXPTS, exist_ok=True)

### Step 6: Move to Handout Directory

In [ ]:
os.chdir(f"/content/{REPO_NAME}")
if os.path.islink('expts') or not os.path.exists('expts'):
    if os.path.islink('expts'): os.unlink('expts')
    os.symlink(DRIVE_EXPTS, 'expts')
!pwd && ls

# Imports
- If your setup was done correctly, you should be able to run the following cell without any issues.

In [ ]:
from hw3lib.data import (
    H3Tokenizer,
    ASRDataset,
    verify_dataloader
)
from hw3lib.model import (
    EncoderDecoderTransformer
)
from hw3lib.utils import (
    create_scheduler,
    create_optimizer,
    plot_lr_schedule
)
from hw3lib.trainers import (
    ASRTrainer,
    ProgressiveTrainer
)
from torch.utils.data import DataLoader
import yaml
import gc
import torch
from torchinfo import summary
import os
import json
import wandb
import pandas as pd
device = "cuda" if torch.cuda.is_available() else "cpu"


# Implementations
- `NOTE`: All of these implementations have detailed specification, implementation details, and hints in their respective source files. Make sure to read all of them in their entirety to understand the implementation details!

## Dataset Implementation
- Implement the `ASRDataset` class in `hw3lib/data/asr_dataset.py`.
- You will have to implement parts of `__init__` and completely implement the `__len__`, `__getitem__` and `collate_fn` methods.
- Run the cell below to check your implementation.


In [ ]:
!python -m tests.test_dataset_asr

## Model Implementations

Overview:

- Implement the `PadMask` and `CausalMask` functions in `hw3lib/model/masks.py`.
- Implement the `PositionalEncoding` and `RotaryPositionalEncoding` classes in `hw3lib/model/positional_encoding.py`.
- Implement the `RMSNorm` class in `hw3lib/model/norms.py`.
- Implement the `MultiHeadAttention` class in `hw3lib/model/attention.py`.
- Implement the `SelfAttentionLayer`, `CrossAttentionLayer`, `SwiGLU` and `FeedForwardLayer` classes in `hw3lib/model/sublayers.py`.
- Implement the `CrossAttentionDecoderLayer` class in `hw3lib/model/decoder_layers.py`.
- Implement the `SelfAttentionEncoderLayer` class in `hw3lib/model/encoder_layers.py`. This will be mostly a copy-paste of the `CrossAttentionDecoderLayer`, without the cross-attention sublayer and without the causal constraint on the input sequence.
- Implement the `EncoderDecoderTransformer` class in `hw3lib/model/transformers.py`.

Implement them in the order below. Each one is used by the next.

### Masks
- Implement the `PadMask` and `CausalMask` functions in `hw3lib/model/masks.py`.
- Run the cells below to check your implementations.

In [ ]:
!python -m tests.test_mask_padding

In [ ]:
!python -m tests.test_mask_causal

### Positional Encoding
- Implement the `PositionalEncoding` class in `hw3lib/model/positional_encoding.py`.
- This is the fixed sinusoidal encoding, added to the embeddings before the first layer.
- Run the cell below to check your implementation.

In [ ]:
!python -m tests.test_positional_encoding

### Rotary Positional Encoding
- Implement the `RotaryPositionalEncoding` class in `hw3lib/model/positional_encoding.py`.
- Unlike `PositionalEncoding`, this one is not added to the embeddings. It rotates the queries and keys inside each attention operation, so it is applied in `MultiHeadAttention` rather than at the top of the model.
- Run the cell below to check your implementation.

In [ ]:
!python -m tests.test_rope

### Normalization
- Implement the `RMSNorm` class in `hw3lib/model/norms.py`.
- Run the cell below to check your implementation.

In [ ]:
!python -m tests.test_norms

### Multi-Head Attention
- Implement the `MultiHeadAttention` class in `hw3lib/model/attention.py`.
- You implemented this in NumPy for `HW3P1`, in `mytorch/nn/multi_head_attention.py`. This is the same module in PyTorch, so keep that file open beside this one. The method names and the `forward` signature are unchanged, `backward` and `init_weights` are no longer needed, and rotary embeddings and QK-Norm are new.
- Run the cell below to check your implementation.

In [ ]:
!python -m tests.test_attention

### Transformer Sublayers
- Implement the `SelfAttentionLayer`, `CrossAttentionLayer`, `SwiGLU` and `FeedForwardLayer` classes in `hw3lib/model/sublayers.py`.
- `NOTE`: `SwiGLU` and `FeedForwardLayer` are separate. `FeedForwardLayer` selects between `SwiGLU` and the GELU network based on `ffn_type`.
- Run the cells below to check your implementations.

In [ ]:
!python -m tests.test_sublayer_selfattention

In [ ]:
!python -m tests.test_sublayer_feedforward

In [ ]:
!python -m tests.test_sublayer_crossattention

### Transformer Cross-Attention Decoder Layer
- Implement the `CrossAttentionDecoderLayer` class in `hw3lib/model/decoder_layers.py`.
- Then run the cell below to check your implementation.


In [ ]:
!python -m tests.test_decoderlayer_crossattention

### Transformer Self-Attention Encoder Layer
- Implement the `SelfAttentionEncoderLayer` class in `hw3lib/model/encoder_layers.py`.
- Then run the cell below to check your implementation.




In [ ]:
!python -m tests.test_encoderlayer_selfattention

### Encoder-Decoder Transformer

- Implement the  `EncoderDecoderTransformer` class in `hw3lib/model/transformers.py`.
- Then run the cell below to check your implementation.

In [ ]:
!python -m tests.test_transformer_encoder_decoder

## Decoding Implementation
- Implement the `generate_greedy` method of the `SequenceGenerator` class in `hw3lib/decoding/sequence_generator.py`.
- Greedy search is required. The trainer's `recognize` method uses it by default, and so does the inference cell at the end of this notebook.
- Run the cell below to check your implementation.

In [ ]:
!python -m tests.test_decoding --mode greedy

### Beam Search
- We highly recommend you to implement the `generate_beam` method of the `SequenceGenerator` class in `hw3lib/decoding/sequence_generator.py`.
- Then run the cell below to check your implementation.
- `NOTE`: This is an optional but highly recommended task for `HW3P2` to ease the journey to high cutoffs!

In [ ]:
!python -m tests.test_decoding --mode beam

## Trainer Implementation
You will have to do some minor in-filling for the `ASRTrainer` class in `hw3lib/trainers/asr_trainer.py` before you can use it.
- Fill in the `TODO`s in the `__init__`.
- Fill in the `TODO`s in the `_train_epoch`.
- Fill in the `TODO`s in the `recognize` method.
- Fill in the `TODO`s in the `_validate_epoch`.
- Fill in the `TODO`s in the `train` method.

`WARNING`: There are no test's for this. Implement carefully!

# Experiments
From this point onwards you may want to switch to a `GPU` runtime.
- `OBJECTIVE`: Optimize your model for `CER` on the test set.

## Config
- You can use the `config.yaml` file to set your config for your ablation study.

---
### Notes:

- Set `tokenization: token_type:` to specify your desired tokenization strategy
- You will need to set the root path to your `hw3p2_data` folder in `data: root:`. This will depend on your setup. For eg.
  - `Local`: `"hw3_data_subset/hw3p2_data"` (the small subset that ships with the handout)
  - `Colab`: `"/content/hw3p2-data/hw3p2_data"`
  - `Kaggle`: the path printed in Kaggle Step 3, for example `"/kaggle/input/datasets/cmu11785/f26-hw3p2-data/hw3p2-data/hw3p2_data"`
  - `PSC`: `"/local/hw3p2-data/hw3p2_data"`
- There's extra configurations in the `optimizer` section which will only be relevant if you decide to use the `create_optimizer` function we've provided in `hw3lib/utils/create_optimizer.py`.
- `BE CAREFUL` while setting numeric values. Eg. `1e-4` will get serialized to a `str` while `1.0e-4` gets serialized to float.

#### T4 configuration notes
- **1k BPE tokens**: transcripts are about 3–4× shorter than with characters, so autoregressive decoding is much faster. The vocab costs only about 0.5M extra parameters.
- **Time reduction ×4 (conv)**: about 1500 frames become about 375, which keeps encoder attention memory and time low on a T4.
- **Size**: d_model 256, 8 encoder / 4 decoder layers, SwiGLU, RoPE, RMSNorm. About 17M parameters (limit is 30M).
- **Regularization**: SpecAugment, dropout 0.1, label smoothing 0.1, CTC weight 0.3.
- **Validation**: each epoch decodes only `val_num_batches` dev batches to save time. The full dev set is decoded at the end.
- **If you hit OOM**: set `batch_size: 16` and `gradient_accumulation_steps: 2`.

In [ ]:
%%writefile config.yaml

Name                      : "Britney Bao"

###### Tokenization ------------------------------------------------------------
tokenization:
  token_type                : "1k"       # [char, 1k, 5k, 10k]
  token_map :
      'char': 'hw3lib/data/tokenizer_jsons/tokenizer_char.json'
      '1k'  : 'hw3lib/data/tokenizer_jsons/tokenizer_1000.json'
      '5k'  : 'hw3lib/data/tokenizer_jsons/tokenizer_5000.json'
      '10k' : 'hw3lib/data/tokenizer_jsons/tokenizer_10000.json'

###### Dataset -----------------------------------------------------------------
data:
  root                 : "/content/hw3p2-data/hw3p2_data"
  train_partition      : "train-clean-100"
  val_partition        : "dev-clean"
  test_partition       : "test-clean"
  subset               : 1.0
  batch_size           : 32
  NUM_WORKERS          : 2
  norm                 : 'global_mvn'
  num_feats            : 80
  feat_dtype           : 'float16'      # store features in fp16 to fit free-Colab RAM

  ###### SpecAugment ---------------------------------------------------------------
  specaug                   : True
  specaug_conf:
    apply_freq_mask         : True
    freq_mask_width_range   : 10
    num_freq_mask           : 2
    apply_time_mask         : True
    time_mask_width_range   : 40
    num_time_mask           : 2

###### Network Specs -------------------------------------------------------------
model:
  input_dim: 80
  time_reduction: 4
  reduction_method: 'conv'

  d_model: 256
  num_encoder_layers: 8
  num_decoder_layers: 4
  num_encoder_heads: 4
  num_decoder_heads: 4
  d_ff_encoder: 1024
  d_ff_decoder: 1024
  encoder_pos_encoding: 'rope'
  decoder_pos_encoding: 'rope'
  norm_type: 'rmsnorm'
  ffn_type: 'swiglu'
  qk_norm: False

  dropout: 0.1
  layer_drop_rate: 0.0
  weight_tying: False

###### Common Training Parameters ------------------------------------------------
training:
  use_wandb                   : True
  wandb_run_id                : "none"   # set to a run ID (from its URL) to resume a run
  gradient_accumulation_steps : 1
  wandb_project               : "HW3P2"
  max_grad_norm               : 1.0      # gradient clipping
  val_num_batches             : 15       # dev batches decoded per epoch (null = full dev set)

###### Loss ----------------------------------------------------------------------
loss:
  label_smoothing: 0.1
  ctc_weight: 0.3

###### Optimizer -----------------------------------------------------------------
optimizer:
  name: "adamw"
  lr: 0.0005
  weight_decay: 0.01
  param_groups: []
  layer_decay:
    enabled: False
    decay_rate: 0.75
  sgd:
    momentum: 0.9
    nesterov: True
    dampening: 0
  adam:
    betas: [0.9, 0.98]
    eps: 1.0e-8
    amsgrad: False
  adamw:
    betas: [0.9, 0.98]
    eps: 1.0e-8
    amsgrad: False

###### Scheduler -----------------------------------------------------------------
scheduler:
  name: "cosine"
  reduce_lr:
    mode: "min"
    factor: 0.5
    patience: 3
    threshold: 0.0001
    threshold_mode: "rel"
    cooldown: 0
    min_lr: 0.0000001
    eps: 1.0e-8
  cosine:
    T_max: 30            # epochs; keep equal to EPOCHS below
    eta_min: 0.000005
    last_epoch: -1
  cosine_warm:
    T_0: 10
    T_mult: 2
    eta_min: 0.0000001
    last_epoch: -1
  warmup:
    enabled: True
    type: "linear"
    epochs: 2
    start_factor: 0.05
    end_factor: 1.0

In [ ]:
with open('config.yaml', 'r') as file:
    config = yaml.safe_load(file)

## Tokenizer

In [ ]:
Tokenizer = H3Tokenizer(
    token_map  = config['tokenization']['token_map'],
    token_type = config['tokenization']['token_type']
)

## Datasets

In [ ]:
train_dataset = ASRDataset(
    partition=config['data']['train_partition'],
    config=config['data'],
    tokenizer=Tokenizer,
    isTrainPartition=True,
    global_stats=None  # Will compute stats from training data
)

# Get the computed global stats from the training set
global_stats = None
if config['data']['norm'] == 'global_mvn':
    global_stats = (train_dataset.global_mean, train_dataset.global_std)
    print(f"Global stats computed from training set.")

val_dataset = ASRDataset(
    partition=config['data']['val_partition'],
    config=config['data'],
    tokenizer=Tokenizer,
    isTrainPartition=False,
    global_stats=global_stats
)

test_dataset = ASRDataset(
    partition=config['data']['test_partition'],
    config=config['data'],
    tokenizer=Tokenizer,
    isTrainPartition=False,
    global_stats=global_stats
)

gc.collect()

## Dataloaders

In [ ]:
train_loader    = DataLoader(
    dataset     = train_dataset,
    batch_size  = config['data']['batch_size'],
    shuffle     = True,
    num_workers = config['data']['NUM_WORKERS'] if device == 'cuda' else 0,
    pin_memory  = True,
    collate_fn  = train_dataset.collate_fn
)

val_loader      = DataLoader(
    dataset     = val_dataset,
    batch_size  = config['data']['batch_size'],
    shuffle     = False,
    num_workers = config['data']['NUM_WORKERS'] if device == 'cuda' else 0,
    pin_memory  = True,
    collate_fn  = val_dataset.collate_fn
)

test_loader     = DataLoader(
    dataset     = test_dataset,
    batch_size  = config['data']['batch_size'],
    shuffle     = False,
    num_workers = config['data']['NUM_WORKERS'] if device == 'cuda' else 0,
    pin_memory  = True,
    collate_fn  = test_dataset.collate_fn
)

gc.collect()

### Dataloader Verification

In [ ]:
verify_dataloader(train_loader)

In [ ]:
verify_dataloader(val_loader)

In [ ]:
verify_dataloader(test_loader)

## Calculate Max Lengths
Calculating the maximum transcript length across your dataset is a crucial step when working with certain transformer models.
-  The positional encodings, sinusoidal or rotary, are precomputed up to a fixed maximum length.
- This maximum length is a hyperparameter that determines:
  - How long of a sequence your model can process
  - The size of your positional encoding matrix
  - Memory requirements during training and inference
- `Requirements`: For this assignment, ensure your positional encodings can accommodate at least the longest sequence in your dataset to prevent truncation.
- `NOTE`: We'll be using the same positional encoding matrix for all sequences in your dataset. Take this into account when setting your maximum length.

In [ ]:
max_feat_len       = max(train_dataset.feat_max_len, val_dataset.feat_max_len, test_dataset.feat_max_len)
max_transcript_len = max(train_dataset.text_max_len, val_dataset.text_max_len, test_dataset.text_max_len)
max_len            = max(max_feat_len, max_transcript_len)

print("="*50)
print(f"{'Max Feature Length':<30} : {max_feat_len}")
print(f"{'Max Transcript Length':<30} : {max_transcript_len}")
print(f"{'Overall Max Length':<30} : {max_len}")
print("="*50)

## Wandb

In [ ]:
wandb.login(key=WANDB_API_KEY)   # 来自最上面的手动填写区

## Training

Every time you run the trainer, it will create a new directory in the `expts` folder with the following structure:
```
expts/
    └── {run_name}/
        ├── config.yaml
        ├── model_arch.txt
        ├── checkpoints/
        │   ├── checkpoint-best-metric-model.pth
        │   └── checkpoint-last-epoch-model.pth
        ├── attn/
        │   └── {attention visualizations}
        └── text/
            └── {generated text outputs}
```


### Training Strategy 1: Cold-Start Trainer

#### Model Load (Default)

In [ ]:
model_config = config['model'].copy()
model_config.update({
    'input_dim': config['data']['num_feats'],
    'max_len': max_len,
    'num_classes': Tokenizer.vocab_size
})

model = EncoderDecoderTransformer(**model_config)

# Get some inputs from the train dataloader
for batch in train_loader:
    padded_feats, padded_shifted, padded_golden, feat_lengths, transcript_lengths = batch
    break

total_param = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert total_param < 30_000_000, f"Total trainable parameters ({total_param}) exceeds 30 million."

model_stats = summary(model, input_data=[padded_feats, padded_shifted, feat_lengths, transcript_lengths])
print(model_stats)

#### Initialize Trainer

If you need to reload the model from a checkpoint, you can do so by calling the `load_checkpoint` method.

```python
checkpoint_path = "path/to/checkpoint.pth"
trainer.load_checkpoint(checkpoint_path)
```


In [ ]:
RUN_NAME = "t4-1k-d256-e8d4-ctc0.3"   # change per experiment
EPOCHS   = config['scheduler']['cosine']['T_max']

trainer = ASRTrainer(
    model=model,
    tokenizer=Tokenizer,
    config=config,
    run_name=RUN_NAME,
    config_file="config.yaml",
    device=device
)

# Resume after a disconnect: uncomment, and also set training.wandb_run_id in config.yaml.
# trainer.load_checkpoint(f"expts/{RUN_NAME}/checkpoints/checkpoint-last-epoch-model.pth")

### Setup Optimizer and Scheduler

You can set your own optimizer and scheduler by setting the class members in the `ASRTrainer` class.
Eg:
```python
trainer.optimizer = optim.AdamW(model.parameters(), lr=config['optimizer']['lr'], weight_decay=config['optimizer']['weight_decay'])
# The trainer steps the scheduler after every optimizer update (every batch by default), so T_max counts updates, not epochs
trainer.scheduler = optim.lr_scheduler.CosineAnnealingLR(trainer.optimizer, T_max=10 * len(train_loader))
```

We also provide a utility function to create your own optimizer and scheduler with the congig and some extra bells and whistles. You are free to use it or not. Do read their code and documentation to understand how it works (`hw3lib/utils/*`).


#### Setting up the optimizer

In [ ]:
trainer.optimizer = create_optimizer(
    model=model,
    opt_config=config['optimizer']
)

#### Creating a test scheduler and plotting the learning rate schedule

In [ ]:
test_scheduler = create_scheduler(
    optimizer=trainer.optimizer,
    scheduler_config=config['scheduler'],
    train_loader=train_loader,
    gradient_accumulation_steps=config['training']['gradient_accumulation_steps']
)

plot_lr_schedule(
    scheduler=test_scheduler,
    num_epochs=20,
    train_loader=train_loader,
    gradient_accumulation_steps=config['training']['gradient_accumulation_steps']
)

#### Setting up the scheduler

In [ ]:
trainer.scheduler = create_scheduler(
    optimizer=trainer.optimizer,
    scheduler_config=config['scheduler'],
    train_loader=train_loader,
    gradient_accumulation_steps=config['training']['gradient_accumulation_steps']
)

#### Train
- One epoch on a T4 takes roughly 8–15 min. 30 epochs is about 5–7 h, which may need one resume on free Colab.

In [ ]:
trainer.train(train_loader, val_loader, epochs=EPOCHS - trainer.current_epoch)

#### Full dev-set evaluation (greedy vs beam) on the best checkpoint

In [ ]:
trainer.load_checkpoint(f"expts/{RUN_NAME}/checkpoints/checkpoint-best-metric-model.pth")
for name, bw in [("greedy", 1), ("beam5", 5)]:
    res = trainer.recognize(val_loader, {'num_batches': None, 'beam_width': bw,
                                         'temperature': 1.0, 'repeat_penalty': 1.0},
                            config_name=name, max_length=max_transcript_len)
    m = trainer._calculate_asr_metrics([r['target'] for r in res], [r['generated'] for r in res])
    print(f"{name:8s} dev CER {m['cer']:.2f}%  WER {m['wer']:.2f}%")

#### Inference



##### W&B Run Tracking for Kaggle Submission

The W&B run is tracked automatically when you train and load
checkpoints using the provided trainer.

Normally, leave `KAGGLE_WANDB_RUN_OVERRIDE = None`.

If you manually replace model weights using
`model.load_state_dict(...)` instead of `trainer.load_checkpoint(...)`,
set `KAGGLE_WANDB_RUN_OVERRIDE` to the exact W&B run that produced
those weights. Manual weight loading does not update the model's
stored W&B association.

Format: `"entity/project/run_id"`

The inference cell ends the trainer's W&B run (`trainer.cleanup()`).
To continue training afterward, first rerun the cells from
`trainer = ASRTrainer(...)` through the scheduler setup. The new
trainer starts a new W&B run. Then reset the override to `None`
before generating a new Kaggle submission, so the trainer can use
the run associated with the new training.

In [ ]:
# Optional W&B provenance override for this Kaggle submission.
# Normally, leave this as None.
KAGGLE_WANDB_RUN_OVERRIDE = None

In [ ]:
# DO NOT CHANGE ================================

# Clear results from any previous inference attempt.
KAGGLE_SUBMISSION_MESSAGE = None
results_df = None

# Check provenance before running inference.
KAGGLE_SUBMISSION_MESSAGE = trainer.build_kaggle_submission_message(
    homework_tag="HW3P2",
    manual_wandb_run=KAGGLE_WANDB_RUN_OVERRIDE,
)
# ===============================================
 
# Define the recognition config: Greedy search
recognition_config = {
    'num_batches': None,
    'temperature': 1.0,
    'repeat_penalty': 1.0,
    'beam_width': 5, # Set to 1 for greedy if beam is not better on dev
}

# Recognize with the config above
config_name = "test"
print(f"Evaluating with {config_name} config")
results = trainer.recognize(test_loader, recognition_config, config_name=config_name, max_length=max_transcript_len)

# Calculate metrics on full batch
generated = [r['generated'] for r in results]
results_df = pd.DataFrame(
    {
        'id': range(len(generated)),
        'transcription': generated,
    }
)

print("Kaggle submission tracking:")
print(KAGGLE_SUBMISSION_MESSAGE)

# Cleanup (will end the active W&B run. To train again, create a new trainer first)
trainer.cleanup()

## Submit to Kaggle

In [ ]:
# DO NOT CHANGE
if (
    globals().get("results_df") is None
    or not globals().get("KAGGLE_SUBMISSION_MESSAGE")
):
    raise RuntimeError("Run the inference cell successfully before submitting.")

results_df.to_csv("results.csv", index=False)

api.competition_submit(
    file_name="results.csv",
    message=KAGGLE_SUBMISSION_MESSAGE,
    competition="f-26-hw-3-p-2-student-competition",
)

##### View the associated W&B run

After submitting, open the competition's **Submissions** tab.
The description beneath each submission shows its associated W&B run:

`HW3P2|WB=entity/project/run_id`

To open that run in W&B, use:

`https://wandb.ai/entity/project/runs/run_id`

This lets you identify which experiment produced each Kaggle submission.


#### **Manual submission (optional)**

If automatic submission is unavailable, download the CSV and submit it manually using one of the options below.

When submitting manually, copy the exact **KAGGLE_SUBMISSION_MESSAGE** printed above into the Kaggle submission description. This information is required to associate the submission with the W&B experiment used to generate it.

##### **Kaggle**

* After running the cell above, the file **`results.csv`** will be saved in:
  **Right sidebar → Output → `kaggle/working/<your repo>/`**
* If the file does not appear right away, click the **refresh icon** in the top-right corner of the Output panel.
* Once the file is visible, **right-click → Download**.
* Upload the downloaded file to the **Kaggle competition submission page**.

---

##### **Colab**

```python
from google.colab import files
files.download("results.csv")
```

##### **PSC**

* Open the left file browser.
* Navigate to:
  `/jet/home/<your_username>/<your repo>/`
* Locate **`results.csv`**.
* Right-click the file and select **Download**.
* If the file does not appear immediately, refresh the file browser.

## 📦 Gradescope Submission Instructions (Read Carefully)

This section guides you through creating and submitting your **final Gradescope code submission**.
Please follow each step in order. Skipping steps may result in an incomplete or invalid submission.

### ⏰ Important Deadlines

* **Kaggle Final Deadline:** See the course schedule
* **Gradescope Code Submission Deadline:**
  **48 hours after** the Kaggle deadline (or the same day as your Slack Kaggle submission, if applicable)

You must submit **both**:

1. A valid Kaggle submission
2. A complete Gradescope code submission zip

Please assign your **final trained model** to the global variable `MODEL`.

We will use this variable to automatically generate a file named
`model_metadata.json`, which records important information about your model (such as parameter count and architecture).

#### 🔴 Important Requirements

* **`MODEL` must be the exact model used for your best Kaggle submission**
* This step is **required** for grading and audit purposes
* If the model does not match your Kaggle submission, your score may be invalidated

#### 🕒 When to Do This

* Run this cell **only after you have finished training** your final model
* Do **not** retrain or modify the model after assigning it to `MODEL`


### Step 1: Confirm Your Final Model

Keep `MODEL = trainer.model` when the current trainer contains
the model used for your best Kaggle submission.

The final submission cell prints the best scored Kaggle submission
and its W&B run. If that submission used a different model,
load its checkpoint into the trainer and rerun the MODEL cell
before recreating the ZIP.

Normally, no change is needed. Changing MODEL does not change
which W&B runs are selected.

In [ ]:
MODEL = trainer.model 

### Step 2: Complete the README Section

Fill in the `README` variable with a brief description of your work:

* **Model**: Architecture and key design choices
* **Training Strategy**: Optimizer, scheduler, loss, etc.
* **Augmentations**: If used (omit if not)
* **Notebook Execution**: Any instructions needed to run your notebook

This README will be included in your submission zip.

In [ ]:
README = """
- **Model**: Model archtiecture description. Anything unique? Any specific architecture shapes or strategies?
- **Training Strategy**: optimizer + scheduler + loss function + any other unique ideas
- **Augmentations**: augmentations if used. If augmentations weren't used, then ignore
- **Notebook Execution**: Any instructions required to run your notebook.
"""

### Step 3: Provide Required Credentials

Ensure the following variables are correctly set:

* `KAGGLE_USERNAME`
* `KAGGLE_API_KEY`: the same `KGAT_` token you used to log in to Kaggle
* `WANDB_API_KEY`
* `WANDB_USERNAME_OR_TEAMNAME`
* `WANDB_PROJECT`

These are used **only** to:

* Fetch your official Kaggle score
* Retrieve top W&B runs associated with your Kaggle submissions for grading and audit purposes.

By default, the backend selects up to 10 distinct W&B runs according to the homework's Kaggle score ordering. You may optionally customize the selected list using **FINAL_WANDB_RUNS_OVERRIDE** (see the next cell).

⚠️ Do **not** share your API keys publicly.

In [ ]:
# @title
# 这些值会自动沿用最上面"手动填写区"填好的内容，这里不用再改
KAGGLE_USERNAME = KAGGLE_USERNAME
KAGGLE_API_KEY  = KAGGLE_API_TOKEN
WANDB_API_KEY   = WANDB_API_KEY
WANDB_USERNAME_OR_TEAMNAME = WANDB_ENTITY
WANDB_PROJECT              = WANDB_PROJECT


##### **Optional W&B Run Selection Override**

By default, the submission backend automatically selects up to **10 distinct W&B runs** associated with your eligible Kaggle submissions, ordered according to the official Kaggle metric for this homework.

**In normal use, leave `FINAL_WANDB_RUNS_OVERRIDE = []`.**

Only modify this list if you intentionally want to change the automatically selected W&B runs after reviewing the selection printed by the final submission cell.

Each entry must use the exact format:

`"entity/project/run_id"`

For example:

```python
FINAL_WANDB_RUNS_OVERRIDE = [
    "my-team/HW3P2-F26/abc123xyz",
    "my-team/HW3P2-F26/def456uvw",
]
```
**Requirements:**
- You may specify at most 10 runs.
- Every run must be associated with one of your eligible Kaggle submissions.
- The run associated with your top-ranked Kaggle submission must remain included.
- Leave the list empty to use the automatic selection.

In [ ]:
# Optional:
# Leave this list empty to use the W&B runs selected automatically
# from your eligible Kaggle submissions.
#
# If overriding, provide at most 10 exact W&B run paths using:
#     "entity/project/run_id"
#
# Example:
# FINAL_WANDB_RUNS_OVERRIDE = [
#     "my-team/HW3P2-F26/abc123xyz",
#     "my-team/HW3P2-F26/def456uvw",
# ]
FINAL_WANDB_RUNS_OVERRIDE = []

### Step 4: Set File Paths Correctly

You must provide absolute path to your **final notebook**

### Platform-specific guidance

**Colab**

* Right-click a file in the left file pane → **Copy path**
* Paths typically start with `/content/...`

**Kaggle**

* Download your notebook (`File → Download Notebook`)
* Upload it via **Upload Input → Upload Model**
* Copy paths from the right sidebar (`/kaggle/working/...`)

**PSC / Linux**

* Locate files under `/jet/home/<your_username>/`
* Use `!ls` to confirm paths

Paste the correct path below.

In [ ]:
# <<< 手动粘贴: 文件 → 在云端硬盘中保存一份副本，然后在左侧文件栏右键该 notebook → 复制路径，粘贴到下面 >>>
NOTEBOOK_PATH = "<<< 手动粘贴: /content/drive/MyDrive/... .ipynb >>>"


### Step 5: (Optional) Additional Files

If you have extra files you want to include (e.g., config files, scripts), add their paths to `ADDITIONAL_FILES`, Otherwise, leave this list empty.

In [ ]:
ADDITIONAL_FILES = []

### Step 6: Generate the Final Submission Zip

Before running the submission cell:

* Ensure `ACKNOWLEDGED = True`
* Double-check all paths and credentials
* Confirm your Kaggle submission exists

Then run the cell below.
This should generate a `HW3P2_final_submission.zip` which includes:
* Your notebook
* `model_metadata.json`
* README
* WandB run exports
* Kaggle metadata
* Academic integrity acknowledgement

In [ ]:
#### DO NOT MODIFY ####
!git clone https://github.com/CMU-IDeeL/HWP2-Submission-Backend-Student-New.git
!mv HWP2-Submission-Backend-Student-New/submission .
!rm -rf HWP2-Submission-Backend-Student-New
from submission.submission_config import SubmissionConfig
from submission.backend_config import BackendConfig, HW3P2_BACKEND_CONFIG
from submission.main import create_submission_zip

create_submission_zip(
    cfg = SubmissionConfig(
        model = MODEL,
        kaggle_username     = KAGGLE_USERNAME,
        kaggle_api_key      = KAGGLE_API_KEY,
        wandb_api_key       = WANDB_API_KEY,
        wandb_entity        = WANDB_USERNAME_OR_TEAMNAME,
        wandb_project       = WANDB_PROJECT,
        wandb_runs_override = FINAL_WANDB_RUNS_OVERRIDE,
        acknowledged        = ACKNOWLEDGED,
        readme              = README,
        notebook_path       = NOTEBOOK_PATH,
        additional_files    = ADDITIONAL_FILES
    ),
    backend_cfg = HW3P2_BACKEND_CONFIG
)

### Step 7: Upload to Gradescope

1. Download `HW3P2_final_submission.zip`
2. Upload it to **Gradescope**
3. Verify the upload succeeds

⚠️ Submitting an incomplete zip or modifying backend code may result in grading penalties or an Academic Integrity Violation.

---

### ✅ Final Checklist

Before submitting, confirm:

* [ ] Kaggle username is valid
* [ ] `model_metadata.json` matches your best run
* [ ] README is complete
* [ ] File paths are correct
* [ ] `ACKNOWLEDGED = True`
* [ ] Final zip downloads successfully